In [0]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

In [0]:
print("Generando Data Mart analítico y exportando a Google Sheets...")

Generando Data Mart analítico y exportando a Google Sheets...


In [0]:
# 1. Crear el Data Mart (Tabla agregada) cruzando Hechos y Dimensiones
query = """
    SELECT 
        d.year,
        d.month,
        p.category,
        COUNT(f.order_id) as total_orders,
        SUM(f.total_item_revenue) as revenue
    FROM workspace.default.gold_fact_sales f
    JOIN workspace.default.gold_dim_date d ON f.date_key = d.date_key
    JOIN workspace.default.gold_dim_product p ON f.product_key = p.product_key
    GROUP BY d.year, d.month, p.category
    ORDER BY d.year DESC, d.month DESC, revenue DESC
"""
df_data_mart = spark.sql(query).toPandas()

# Mostrar las primeras 5 filas para verificar que funcionó antes de enviarlo a Sheets
display(df_data_mart.head())

year,month,category,total_orders,revenue
2018,9,moveis_cozinha_area_de_servico_jantar_e_jardim,1,166.46
2018,8,beleza_saude,849,137206.0700000002
2018,8,relogios_presentes,416,79391.86000000003
2018,8,cama_mesa_banho,658,75207.83000000003
2018,8,utilidades_domesticas,629,74238.88999999998


In [0]:
# 2. Configurar la API de Google Sheets
scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
# Ruta donde subiste tus credenciales en Databricks
creds = ServiceAccountCredentials.from_json_keyfile_name('/Volumes/workspace/default/olist_raw_data/credenciales.json', scope)
client = gspread.authorize(creds)

In [0]:
# 3. Conectar al Sheet y cargar los datos
sheet_name = "ecommerce-lakehouse" 
sheet = client.open(sheet_name).sheet1

# Limpiar la hoja antes de cargar nueva data (Idempotencia)
sheet.clear()

# Insertar datos (Incluyendo encabezados)
sheet.update([df_data_mart.columns.values.tolist()] + df_data_mart.values.tolist())

{'spreadsheetId': '1Gm5DG5xuOzuBfKo-4e2d0xWaMHkF5NGipvniH9oSn5A',
 'updatedRange': "'Hoja 1'!A1:E1284",
 'updatedRows': 1284,
 'updatedColumns': 5,
 'updatedCells': 6420}

In [0]:
print(f"✅ Exportación exitosa. {len(df_data_mart)} filas agregadas enviadas a {sheet_name}.")

✅ Exportación exitosa. 1283 filas agregadas enviadas a ecommerce-lakehouse.
